In [0]:
import sys
from superstore_logger import get_superstore_logger, log_event, create_master_run_id
import os

# -----------------------------
# Add shared utilities path
# -----------------------------
sys.path.append("/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/superstore_shared_utilities")

# -----------------------------
# Read job parameters (from DAB)
# -----------------------------
dbutils.widgets.text("SUPERSTORE_ENV", "dev")
dbutils.widgets.text("SUPERSTORE_PIPELINE_NAME", "superstore_etl_pipeline")
dbutils.widgets.text("SUPERSTORE_PIPELINE_VERSION", "1.0")

env = dbutils.widgets.get("SUPERSTORE_ENV")
pipeline_name = dbutils.widgets.get("SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.widgets.get("SUPERSTORE_PIPELINE_VERSION")

# -----------------------------
# Set runtime environment variables
# -----------------------------
os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

# -----------------------------
# Initialize logger
# -----------------------------
logger_master = get_superstore_logger("superstore_master_orchestrator")

try:
    # -----------------------------
    # Create Master Run ID
    # -----------------------------
    master_run_id: str = create_master_run_id()  # Explicit type annotation
    log_event(
        logger_master,
        "INFO",
        "Master Run ID for superstore medallion lakehouse pipeline successfully created",
        master_run_id=master_run_id,
        layer=None
    )

except Exception as e:
    # -----------------------------
    # Handle failure in master ID creation
    # -----------------------------
    master_run_id: str = "MASTER_ID_CREATION_FAILED"
    log_event(
        logger_master,
        "ERROR",
        f"Failed to create Master Run ID: {str(e)}",
        master_run_id=master_run_id,
        layer=None
    )
    # Optionally exit immediately if this is critical
    import sys
    sys.exit(f"Critical: Could not generate master_run_id. Error: {e}")

# -----------------------------
# Log pipeline start
# -----------------------------
log_event(
    logger_master,
    "INFO",
    "Initiating Superstore medallion lakehouse Pipeline using Delta Lake Medallion Architecture",
    master_run_id=master_run_id,
    layer=None
)


# -----------------------------
# Share master_run_id with DAG tasks
# -----------------------------
dbutils.jobs.taskValues.set(key="master_run_id", value=master_run_id)

dbutils.jobs.taskValues.set(key="SUPERSTORE_ENV", value=env)
dbutils.jobs.taskValues.set(key="SUPERSTORE_PIPELINE_NAME", value=pipeline_name)
dbutils.jobs.taskValues.set(key="SUPERSTORE_PIPELINE_VERSION", value=pipeline_version)

print(f"SUPERSTORE DATA PLATFORMS")
print(f"ENV: {env}")
print(f"PIPELINE_NAME: {pipeline_name}")
print(f"PIPELINE_VERSION: {pipeline_version}")
print(f"MASTER_RUN_ID: {master_run_id}")